# End-to-End RR-CTP Upsampler (RR-CTP: Rotation-Routed Conditional Tensor Product)

## Clean Math and System Architecture Note

**Problem setting**: Super-resolution of crystallographic orientation maps (EBSD). The input is a low-resolution quaternion field from crystal quotient space, and the output is a high-resolution quaternion field with clean grain interiors and boundary-aware discontinuities.

**Goal of this notebook**: define one clean end-to-end upsampler architecture that starts from the LR encoded feature field, uses invariant selector logic to choose semiglobal context, builds HR features through equivariant expert heads, and only decodes to quaternions at the end.

**Locked design choices in this version**:
- Upsampling factor: $4\times$
- Hard LR context window: $5\times5$
- Total expert count: $K=12$
- Active experts per pixel: top-2 by default
- HR seed is built from routed experts
- Gating acts inside attention before context pooling
- Optional shared TP refinement is applied only after the expert-built seed exists

**Main design principle**: the upsampler must decide grain ownership before quaternion decode. It should use semiglobal LR evidence to choose the right orientation hypothesis for each HR pixel, and it should only allow interaction when that hypothesis is trustworthy.

**Symmetry rule in this version**: selector logic is allowed to be a scalar decision pathway, but for fixed experts its outputs must be invariant; all feature-valued maps that produce encoded HR features must be equivariant.


## 1. System View

**Step goal**: show the complete end-to-end model in one system-level pipeline.

Dataflow to track through later sections: $F^{\mathrm{LR}}$ is the LR trunk output handed to the upsampler; inside the upsampler the key carried tensors are the query anchor $q_u^0$, candidate bank $B_u$, expert contexts $c_u^{(k)}$, routed seed $s_u$, and final encoded HR feature $y_u$. Each later section refines one arrow in this pipeline rather than introducing a disconnected model fragment.

Overall flow:

$$Q^{\mathrm{LR}} \;\xrightarrow{\mathrm{Encoder}_{\mathrm{mat}}}\; F_0^{\mathrm{LR}} \;\xrightarrow{\mathrm{LR\ trunk}}\; F^{\mathrm{LR}} \;\xrightarrow{\mathrm{RR\text{-}CTP\ Upsampler}}\; Y^{\mathrm{HR}} \;\xrightarrow{\mathrm{Decoder}_{\mathrm{mat}}}\; \hat Q^{\mathrm{HR}}.$$

Inside the RR-CTP upsampler, each HR pixel follows this sub-pipeline:

$$\text{parent LR feature conditioned by subpixel phase} \;\to\; q_u^0 \;\to\; 5\times5\ \text{candidate bank} \;\to\; \text{invariant gated scoring / routing} \;\to\; \text{expert contexts} \;\to\; \text{expert-specific equivariant seed } s_u \;\to\; \text{optional TP refinement} \;\to\; y_u.$$

Module roles:
- `Encoder_mat`: converts quaternions to material-aware irreps features.
- `LR trunk`: builds better LR evidence before any HR decision is made.
- `RR-CTP Upsampler`: performs semiglobal candidate selection, invariant routing, equivariant seed construction, and selective refinement.
- `Decoder_mat`: maps final HR encoded features back to passive quaternions.

**Step conclusion**: the upsampler is not a generic interpolation block. It is the decision-making core between the LR feature trunk and the final quaternion decoder.


## 2. Representation Space and Material Awareness

**Step goal**: define the feature space in which the entire SR pipeline operates.

Dataflow role of this section: it does not introduce a new algorithmic step; instead it fixes the representation spaces in which every later tensor lives. When later sections mention $f_i$, $q_u^0$, $c_u^{(k)}$, $s_u$, or $y_u$, this section tells the reader which of $V_{a1}$ or $V_{\mathrm{up}}$ those tensors belong to and which quantities must be invariant scalars.

For the chosen material, the encoded feature at any LR or HR location lives in
$$V_{\mathrm{mat}} = \bigoplus_{j=1}^{J} m_j\,V^{(l_j)}.$$

In this repository, `irreps_a1` means the active local-isometry feature subspace, not automatically the SO(3) scalar channel. Representative examples are:
- FCC (Oh): $\mathrm{irreps\_a1} = 1\times 4e$ with dimension $9$
- FCC full upsampler space: $\mathrm{irreps\_full} = 1\times 2e + 1\times 4e$ with dimension $14$
- HCP (D6): $\mathrm{irreps\_a1} = 2\times 2e + 1\times 4e + 1\times 6e$ with dimension $32$

The correct rotation action is material-dependent:
$$\rho_{\mathrm{mat}}(R) = \bigoplus_j \left(I_{m_j} \otimes D^{(l_j)}(R)\right).$$

For the intended implementation, the RR-CTP block runs in a chosen upsampler feature space $V_{\mathrm{up}}$. In FCC we intend
$$V_{\mathrm{up}} = 1\times 2e \oplus 1\times 4e,$$
while the final decoder still consumes the projected `a1` output. The upsampler therefore lives in a richer equivariant space than the final decode space.

The design rule is:
- Feature-valued maps such as query anchors, expert contexts, seed proposals, and refined HR outputs must remain in $V_{\mathrm{up}}$ and be equivariant.
- Selector outputs such as candidate logits, expert logits, top-$k$ choices, routing weights, and refinement gates are scalars. For fixed experts, these scalars must be invariant under the material symmetry action.

**Why this matters**:
- The same architecture can be reused across FCC, HCP, and future materials.
- The selector path can make hard decisions without breaking symmetry, provided those decisions are driven by invariants.
- Expert identity should be expressed through expert-specific equivariant heads, not through unconstrained dense feature-space transforms.

**Step conclusion**: the intended upsampler is a material-aware equivariant feature pipeline controlled by an invariant scalar decision pathway.


## 3. End-to-End Mathematical Definition

**Step goal**: write the full model from LR quaternion input to HR quaternion output.

Dataflow role of this section: it names the major tensors passed between the large blocks of the model. The later sections then unpack the internal dataflow of the RR-CTP upsampler that maps $F^{\mathrm{LR}}$ to $Y^{\mathrm{HR}}$.

Let the LR input field be
$$Q^{\mathrm{LR}} = \{q_i^{\mathrm{LR}}\}_{i=1}^{N_{\mathrm{LR}}}.$$

Encode each LR quaternion:
$$f_i^{a1,0} = \mathrm{Encoder}_{\mathrm{mat}}\!\left(q_i^{\mathrm{LR}}\right) \in V_{a1}.$$

Process the encoded LR field with the existing LR backbone and lift it into the upsampler feature space:
$$F^{\mathrm{LR}} = \{f_i\}_{i=1}^{N_{\mathrm{LR}}} = \mathrm{LRTrunk}\!\left(\{f_i^{a1,0}\}\right), \qquad f_i \in V_{\mathrm{up}}.$$

Then the RR-CTP upsampler maps the LR feature field to an HR feature field in the same equivariant upsampler space:
$$Y^{\mathrm{HR}} = \{y_u\}_{u=1}^{N_{\mathrm{HR}}} = \mathrm{Upsampler}_{\mathrm{RR\text{-}CTP}}\!\left(F^{\mathrm{LR}}\right), \qquad y_u \in V_{\mathrm{up}}.$$

Project the HR feature field back to the decoder input space and decode to passive quaternions:
$$\tilde y_u = \mathrm{Proj}_{a1}(y_u) \in V_{a1}, \qquad \hat q_u^{\mathrm{HR}} = \mathrm{Decoder}_{\mathrm{mat}}\!\left(\tilde y_u\right).$$

So the complete model is
$$\hat Q^{\mathrm{HR}} = \mathrm{Decoder}_{\mathrm{mat}}\!\Big(\mathrm{Proj}_{a1}\big(\mathrm{Upsampler}_{\mathrm{RR\text{-}CTP}}(\mathrm{LRTrunk}(\mathrm{Encoder}_{\mathrm{mat}}(Q^{\mathrm{LR}})))\big)\Big).$$

For FCC, the intended concrete spaces are
$$V_{a1} = 1\times 4e, \qquad V_{\mathrm{up}} = 1\times 2e \oplus 1\times 4e.$$

**Step conclusion**: all interpolation, grain assignment, and selective interaction happen in encoded irreps space before the final decode step, with RR-CTP operating in the richer upsampler feature space.


## 4. Semiglobal 5x5 Candidate Bank

**Step goal**: define the context support used by the upsampler after the LR trunk.

Incoming data from Section 3: the LR trunk has already produced $F^{\mathrm{LR}} = \{f_i\}$ in $V_{\mathrm{up}}$. This section reorganizes those LR features into the per-HR-pixel candidate bank $B_u$ and visibility gate terms $G_{u,i}$, which become the inputs to the query-to-bank interaction in Sections 5 and 6.

For each HR pixel center $u$, determine its parent LR location and gather a fixed LR window
$$\mathcal W_u = \{i : x_i \text{ lies in a } 5\times5 \text{ LR window around the parent of } u\}. $$

This is a fixed LR support. For $4\times$ SR, it corresponds to a $20\times20$ HR-equivalent field of view, which keeps the candidate bank local while still letting each HR query compare nearby competing evidence.

The window is treated as a candidate bank, not as one pre-averaged context vector:
$$B_u = \{(f_i,\; \Delta x_{u,i})\}_{i\in\mathcal W_u}. $$

A learnable visibility gate determines the effective support inside the hard window:
$$G_{u,i} = G_{\mathrm{win}}(u,i)\; G_{\mathrm{boundary}}(u,i)\; G_{\mathrm{aff}}(u,i).$$

Interpretation of each factor:
- $G_{\mathrm{win}}$: learnable spatial preference inside the $5\times5$ support.
- $G_{\mathrm{boundary}}$: optional boundary-aware suppression factor. If LR grain labels or trusted boundary side information are not provided, this term defaults to $1$.
- $G_{\mathrm{aff}}$: favors candidates whose invariant features are compatible with the query.

So the default unlabeled setting is
$$G_{u,i} = G_{\mathrm{win}}(u,i)\; G_{\mathrm{aff}}(u,i),$$
and explicit same-grain masking is only an optional supervised extension.

**Important rule**: do not collapse the full $5\times5$ window into one mixture before expert routing. The experts must first score separate candidates and only then merge them in a controlled way.

Data passed forward: for each HR pixel we now have candidate features $\{f_i\}_{i\in\mathcal W_u}$, relative offsets $\Delta x_{u,i}$, and visibility gates $G_{u,i}$.

**Step conclusion**: the upsampler uses a fixed semiglobal candidate bank with learnable effective support, which provides long-range context without uncontrolled cross-grain averaging. In the default unlabeled regime, boundary awareness is indirect rather than label-supervised.


## 5. Raw HR Query Anchor

**Step goal**: define the lightweight HR query used to interrogate the semiglobal candidate bank.

Incoming data from Section 4: for HR location $u$ we already know its parent LR location and the candidate bank around that parent. This section adds the local query tensor $q_u^0$, which will be compared against the bank entries in Section 6.

For each HR pixel $u$, build a conservative query anchor
$$q_u^0 = Q_{\mathrm{eq}}\!\left(f_{\mathrm{parent}(u)},\; e_{\mathrm{phase}}(u)\right) \in V_{\mathrm{up}},$$
where $e_{\mathrm{phase}}(u)$ is a learned subpixel phase embedding represented as invariant scalar conditioning. For $4\times$ SR, the HR grid has $4\times4=16$ phase positions inside each parent LR cell.

This anchor is intentionally cheap. It uses local evidence from the parent LR feature and the HR subpixel phase as a conditioning signal, but it is not yet the final HR seed. Its job is to act as the query that asks the semiglobal LR bank which grain hypothesis is most plausible.

The key constraint is that $Q_{\mathrm{eq}}$ must be equivariant in its feature argument. Phase information is allowed to modulate the anchor only through invariant scalar channels, gates, or coefficients. It is not added directly to the equivariant feature coordinates as if both lived in the same modality or representation space.

Invariant query summaries can then be formed from blockwise norms and alignments. Here $a$ denotes the query feature being summarized, typically $a = q_u^0$, and $b$ denotes the comparison feature, typically one LR candidate feature $b = f_i$ or an aggregated bank summary derived from the candidate bank. For a general decomposition $V_{\mathrm{up}} = \bigoplus_j m_j V^{(l_j)}$ we use
$$\mathrm{InvStats}(a,b) = \mathrm{concat}_j\Big[\|a^{(j)}\|_2,\; \|b^{(j)}\|_2,\; \langle a^{(j)}, b^{(j)}\rangle\Big].$$

For FCC in particular, with $V_{\mathrm{up}} = 1\times2e \oplus 1\times4e$, this becomes a blockwise summary over the $2e$ and $4e$ parts of the feature.

**Why this matters**:
- The anchor provides a stable starting point for all experts.
- It keeps the seed conservative until semiglobal evidence has been examined.
- It lets the model use phase-specific behavior without leaving encoded space.

Data passed forward: each HR pixel now carries its query anchor $q_u^0$, the phase-conditioning signal $e_{\mathrm{phase}}(u)$, and the blockwise invariant-statistics recipe used to compare $q_u^0$ against bank features.

**Step conclusion**: each HR pixel starts from a cheap equivariant local query anchor, which is used for routing and context selection but is not itself trusted as the final output.


## 6. Gated Expert Attention and Context Selection

**Step goal**: define how experts examine the $5\times5$ candidate bank and construct context.

Incoming data from Sections 4-5: each HR pixel now has a query $q_u^0$, a candidate bank $B_u$, relative offsets $\Delta x_{u,i}$, and visibility gates $G_{u,i}$. This section turns those into expert contexts plus the scalar routing decisions that control the next two sections.

For each expert $k \in \{1,\ldots,K\}$ and each candidate $i \in \mathcal W_u$, compute a gated score
$$\omega_{u,i}^{(k)} = \mathrm{Score}_k\!\left(\mathrm{InvStats}(q_u^0, f_i),\; \Delta x_{u,i},\; e_{\mathrm{phase}}(u)\right) + \log\!\big(G_{u,i}+\varepsilon\big).$$

Normalize over the candidate bank:
$$\alpha_{u,i}^{(k)} = \mathrm{softmax}_{i\in\mathcal W_u}\big(\omega_{u,i}^{(k)}\big).$$

Then form the expert-specific semiglobal context
$$c_u^{(k)} = \sum_{i\in\mathcal W_u} \alpha_{u,i}^{(k)} f_i.$$

Because each $\alpha_{u,i}^{(k)}$ is an invariant scalar and each $f_i$ lives in $V_{\mathrm{up}}$, the pooled context $c_u^{(k)}$ remains equivariant.

The per-pixel router uses invariant statistics derived from the query and the bank summary:
$$[\ell_{u,1},\ldots,\ell_{u,K},\ell_{u,g}] = \mathrm{Router}\!\left(z_u\right).$$

With the default policy, choose
$$\mathcal A_u = \mathrm{top\text{-}2}(\ell_{u,1},\ldots,\ell_{u,K}), \qquad \pi_{u,k} = \mathrm{softmax}_{k\in\mathcal A_u}(\ell_{u,k}).$$

The router outputs are scalar logits. For fixed experts, the requirement is not that the router itself be an equivariant feature map, but that these logits be invariant so that the same experts are selected before and after a symmetry action.

**Key interpretation**:
- Experts distinguish candidate orientations first.
- Only after that discrimination do they build a weighted context.
- Because the gate is injected inside the logits, unsafe candidates are suppressed before context pooling, not after.
- Hard `top-k` routing is symmetry-safe provided the underlying routing logits are invariant.

Data passed forward: for each HR pixel we now have expert contexts $c_u^{(k)}$, selected experts $\mathcal A_u$, routing weights $\pi_{u,k}$, and router gate information that will become the refinement gate in Section 8.

**Step conclusion**: the expert stage is an invariant gated candidate-selection process over the semiglobal bank, not a blind smoothing operator.


## 7. Expert-Built HR Seed

**Step goal**: define the HR seed as the routed output of the experts.

Incoming data from Section 6: the model now knows which experts are active, how strongly each selected expert should contribute, and what context each expert extracted from the bank. This section combines $q_u^0$, $c_u^{(k)}$, $e_{\mathrm{phase}}(u)$, and $\pi_{u,k}$ to produce the routed seed $s_u$, which becomes the direct input to Section 8.

Each active expert produces an expert-specific seed proposal
$$s_u^{(k)} = \mathrm{SeedEq}_k\!\left(q_u^0,\; c_u^{(k)},\; e_{\mathrm{phase}}(u)\right) \in V_{\mathrm{up}}.$$

The final HR seed is the routed blend of the active experts:
$$s_u = \sum_{k\in\mathcal A_u} \pi_{u,k}\, s_u^{(k)}.$$

This is the critical architectural decision. The seed is not obtained by plain bilinear interpolation or by a generic transpose convolution alone. It is formed only after the experts have examined semiglobal LR evidence and selected a grain-conditioned context.

Expert identity enters through the expert-specific equivariant seed heads themselves. We do not rely on unconstrained dense feature-space transforms to create expert specialization.

Because the seed proposals are equivariant and the routing weights $\pi_{u,k}$ are invariant scalars, the mixed seed $s_u$ remains equivariant.

**Why this matters**:
- Interior pixels receive a strong, consistent grain-conditioned seed.
- Boundary pixels can choose between competing orientation hypotheses before any hard decode happens.
- Junctions and ambiguous zones can still produce a conservative seed even when refinement should be weak.

Data passed forward: the upsampler now carries the routed seed $s_u$ together with the still-active contexts $c_u^{(k)}$ and routing weights $\pi_{u,k}$.

**Step conclusion**: the seed itself is a routed equivariant decision in encoded space, not a naive spatial lift.


## 8. Shared TP Refinement After Seed Formation

**Step goal**: optionally refine the expert-built seed using a shared equivariant interaction block.

Incoming data from Sections 6-7: the seed stage has already produced $s_u$, while the routing stage still supplies the selected expert contexts $c_u^{(k)}$, routing weights $\pi_{u,k}$, and refinement-gate information. This section converts those into the final encoded HR feature $y_u$ that will later be projected and decoded.

Once the HR seed $s_u$ exists, use the same active expert contexts for one shared TP refinement stage:
$$\delta_u^{(k)} = \mathrm{TP}_{\mathrm{shared}}\big(s_u,\; c_u^{(k)}\big), \qquad \Delta_u = \sum_{k\in\mathcal A_u} \pi_{u,k}\, \delta_u^{(k)}.$$

The router also predicts a scalar refinement-trust gate
$$g_u = \sigma(\ell_{u,g}).$$

The final HR encoded feature is
$$y_u = s_u + g_u\,\Delta_u.$$

This creates a two-stage safety mechanism:
1. Candidate-level gating inside the expert attention bank.
2. Final scalar gating on whether the post-seed TP refinement should be trusted.

Because $g_u$ and $\pi_{u,k}$ are invariant scalars while $s_u$, $c_u^{(k)}$, and $\delta_u^{(k)}$ are equivariant features, the final output
$$y_u = s_u + g_u\,\Delta_u$$
remains equivariant.

**Interpretation**: refinement is allowed only after grain ownership has been estimated. When the local situation is ambiguous, the model can fall back toward the safer expert-built seed.

Data passed forward: the RR-CTP block now outputs the final encoded HR feature $y_u \in V_{\mathrm{up}}$, ready for interpretation, training, projection to $V_{a1}$, and quaternion decode.

**Step conclusion**: TP remains useful, but only as a selective refinement on top of an already-routed seed, with the selector path kept invariant and the feature path kept equivariant.


## 9. Expected Behavior by Region

**Step goal**: make the intended regional behavior explicit.

This section is interpretive rather than generative: it does not create new tensors, but explains how the previously defined quantities $G_{u,i}$, $\pi_{u,k}$, $s_u$, and $g_u$ are expected to behave in different spatial regimes.

| Region | Candidate visibility | Expert behavior | Refinement gate | Effective outcome |
|--------|----------------------|-----------------|-----------------|------------------|
| Grain interior | Broad, permissive | One expert usually dominates | High | Strong grain-consistent seed and strong refinement |
| Two-grain boundary | Cross-boundary candidates suppressed indirectly by affinity and routing, or explicitly if boundary side information exists | Two experts may compete | Moderate | Expert choice is important; refinement only if context is still trustworthy |
| Triple junction | Visibility becomes sparse and selective | Expert disagreement rises | Low to moderate | Conservative seed with weak or no refinement |
| Noisy / uncertain zone | Bank confidence is weak | Router confidence is weak | Low | Model falls back toward the safer seed path |

**Operational meaning**:
- In interiors, the upsampler behaves like a confident equivariant refiner.
- Near boundaries, it behaves like a grain-hypothesis selector first and a refiner second.
- In ambiguous zones, it behaves conservatively to avoid destructive cross-grain interaction.
- In the default unlabeled setting, boundary handling is emergent from invariant affinity and expert competition rather than explicit same-grain masking.

**Step conclusion**: the same architecture changes behavior by region through visibility gating, expert competition, and the final refinement gate.


## 10. Training Objective and Module Budget

**Step goal**: connect the architecture to the loss design, while distinguishing the current minimal implementation from the fuller future version.

Incoming data from the pipeline: after Section 7, the current minimal implementation already has a routed HR encoded feature prediction. If Section 8 is later enabled, the pipeline will instead produce a post-refinement HR encoded feature before projection and decode. The loss design should therefore follow the actual endpoint that the implemented model produces, rather than the fullest possible future architecture.

There are two regimes to keep separate:
- **Minimal architecture now**: `encoder -> seed-stage RR-CTP -> feature-space supervision`, with the decoder used only for inference or visualization.
- **Full version later**: `encoder -> seed-stage RR-CTP -> optional shared TP refinement -> optional projection/decode supervision`, with room for auxiliary router or symmetry regularizers if the implemented behavior shows that they are needed.

For the model currently wired in `models/SR_rrctp_semiglobal.py`, the active training objective is just
$$\mathcal L = \mathcal L_{\mathrm{feat}} = \|\hat F^{\mathrm{HR}} - F^{\mathrm{HR}}\|_2^2,$$
implemented as mean-squared error over the encoded HR feature vectors.

Concretely, in the minimal architecture:
- The LR quaternions are encoded to LR features.
- The HR quaternions are encoded to target HR features.
- The RR-CTP block predicts HR features from the LR features.
- Training minimizes feature-space MSE between predicted and target HR features.

The reason this minimal objective is enough for the present code is architectural:
- The current model stops at the routed HR seed, so the seed prediction is already the final trained encoded output.
- The optional shared TP refinement stage from Section 8 is not yet active, so there is no refinement gate or post-seed correction branch to regularize.
- We are not supplying grain labels or boundary side information, so no supervised boundary-side loss has a meaningful target.
- The decoder is intentionally outside the training path, which avoids making every optimization step depend on the expensive decoder search.
- The router and experts are trained indirectly through the main reconstruction signal; until there is a demonstrated collapse or degeneracy, extra router/expert regularizers are not justified.

A clean term-by-term rule is:

| Loss term | Minimal architecture now | Full version later | Why |
|---|---|---|---|
| $\mathcal L_{\mathrm{feat}}$ feature-space MSE | **Needed** | **Still needed** | It directly supervises the encoded HR field that the RR-CTP block actually produces, and it is the cheapest stable training signal. |
| $\mathcal L_{\mathrm{quat}}$ decoded quaternion task loss | **Not needed** | Optional | In the minimal model the decoder is not on the training path. In the fuller version it is only worth adding if we deliberately want supervision after decode and accept the extra cost. |
| $\mathcal L_{\mathrm{seed}}$ separate seed-only auxiliary loss | **Not needed separately** | Optional | In the minimal model the seed is already the final encoded output, so this would duplicate $\mathcal L_{\mathrm{feat}}$. Once refinement exists, a seed-specific auxiliary loss can keep the seed itself well-posed. |
| $\mathcal L_{\mathrm{gate}}$ refinement-gate regularization | **Not needed** | Optional or needed if refinement is used | There is no refinement gate in the minimal architecture. If the full version uses $y_u = s_u + g_u\Delta_u$, then gate regularization may be useful to stop always-on refinement. |
| $\mathcal L_{\mathrm{bal}}$ router load balancing | **Not needed by default** | Optional | The current router is trained only through reconstruction. Add balancing only if experts collapse and some experts never get used. |
| $\mathcal L_{\mathrm{sep}}$ expert separation/diversity | **Not needed by default** | Optional | Do not force diversity unless the experts actually become redundant in practice; otherwise this is just an extra hyperparameter. |
| $\mathcal L_{\mathrm{eq}}$ explicit symmetry-consistency regularization | **Not needed by default** | Optional | It requires extra transformed forward passes. Only add it if architectural symmetry alone is not strong enough in practice. |
| Boundary-side or grain-label loss | **Not needed** | Only if labels are supplied | In the current unlabeled setting there is no supervised target for same-grain or side-aware behavior. |
| Decoder or decode-consistency loss | **Not needed** | Optional | The decoder is presently an inference tool, not an optimization target. This only becomes relevant if decode quality itself is put inside training. |

So the relevant monitored quantities in the current implementation are simply:
- Train feature MSE.
- Validation feature MSE.
- Learning-rate schedule.

All other loss terms should stay out of the optimization until the corresponding mechanism actually exists in the implemented model and there is a concrete failure mode that the new term is meant to fix.

Parameter count is still not summarized here as a fixed closed-form number, because the authoritative budget depends on the final chosen hidden widths, expert count, and whether the optional refinement path is added later.

**Important note**: the $5\times5$ window mainly affects compute and memory, not parameter count, because the scoring and routing weights are shared across candidates in the bank.

**Step conclusion**: the notebook now matches the current code path and the intended extension path: use only feature-space MSE for the present seed-stage model, and promote additional losses only when the fuller architecture actually contains the branch, gate, supervision, or failure mode that those terms are meant to address.


## 11. System Architecture Modules and Tensor Shapes

**Step goal**: translate the math into concrete model components that can be inserted after the LR blocks.

This section converts the mathematical dataflow of Sections 3-8 into module boundaries and tensor contracts. The tensor list below should be read in pipeline order, so each line becomes the input shape for the next computation.

Recommended new module boundary after the LR trunk:
- `RRCTPSemiglobalUpsampler`
- `PhaseEmbedding4x4`
- `SharedWindowGate5x5`
- `InvariantStatBuilder`
- `InvariantCandidateScorer`
- `InvariantRouter`
- `EquivariantQueryAnchor`
- `ExpertEquivariantSeedHead`
- `SharedTPRefine`
- `FinalProjToA1`

Useful tensor shapes for batch size $B$, LR size $H\times W$, HR size $(4H)\times(4W)$, upsampler feature dimension $C_{\mathrm{up}} = \dim(V_{\mathrm{up}})$, decoder feature dimension $C_{a1} = \dim(V_{a1})$, and candidate count $M=5\times5=25$:
- LR trunk output: $(B, HW, C_{\mathrm{up}})$
- HR query anchors: $(B, 16HW, C_{\mathrm{up}})$
- Candidate bank features: $(B, 16HW, M, C_{\mathrm{up}})$
- Candidate visibility gates: $(B, 16HW, M)$
- Invariant candidate statistics: $(B, 16HW, M, D_{\mathrm{inv}})$
- Expert candidate logits: $(B, 16HW, K, M)$ before top-k pruning
- Routed expert contexts: $(B, 16HW, K, C_{\mathrm{up}})$
- Expert seeds: $(B, 16HW, K, C_{\mathrm{up}})$
- Final routed seed: $(B, 16HW, C_{\mathrm{up}})$
- Final HR encoded output in upsampler space: $(B, 16HW, C_{\mathrm{up}})$
- Projected decoder input: $(B, 16HW, C_{a1})$

**Implementation rule**: the existing decoder stays unchanged. The new work happens between the LR trunk output and the decoder input, with RR-CTP operating in $V_{\mathrm{up}}$ and only the final projection moving back to $V_{a1}$.

**Step conclusion**: the proposed math maps cleanly to a bounded set of modules and tensor contracts that can be implemented after the LR backbone.


## 12. Clean Integration Guide After the LR Blocks

**Step goal**: provide a direct step-by-step implementation order for the end-to-end upsampler.

This section linearizes the earlier dataflow into implementation order. Each numbered step consumes the tensors defined in the preceding steps and produces the tensors named in the next one, so the section can be used as a build checklist.

Implementation sequence:
1. Keep the current encoder, LR trunk, final projection, and decoder boundaries unchanged.
2. Remove the plain HR upsample stage as the main decision maker.
3. Insert `RRCTPSemiglobalUpsampler` immediately after the final LR block output $F^{\mathrm{LR}}$, operating in $V_{\mathrm{up}}$.
4. For every HR pixel, gather a $5\times5$ LR candidate bank around the parent LR location.
5. Build an equivariant query anchor $q_u^0$ from the parent LR feature, conditioned by the scalar $4\times4$ phase embedding rather than by directly adding phase vectors to feature coordinates.
6. Compute candidate visibility gates $G_{u,i}$ inside the bank, with $G_{\mathrm{boundary}} \equiv 1$ by default unless boundary side information is provided.
7. Build invariant candidate statistics from $q_u^0$, each candidate feature, relative offset, and phase.
8. For each of the $12$ experts, score candidates with invariant gated logits and form expert-specific contexts.
9. Build invariant router logits, apply top-2 expert selection, and obtain invariant routing weights $\pi_{u,k}$ and invariant refinement gate $g_u$.
10. Produce expert seed proposals with expert-specific equivariant seed heads; do not use unconstrained dense expert-frame rotations as the core specialization mechanism.
11. Blend the selected expert seed proposals to produce the HR seed $s_u$.
12. Apply one shared TP refinement stage to produce $\Delta_u$, then modulate it with the scalar refinement gate $g_u$.
13. Output $y_u = s_u + g_u\Delta_u$ as the final HR encoded feature in $V_{\mathrm{up}}$.
14. Project to $V_{a1}$ and decode the HR encoded field with the existing quaternion decoder.

Clean default policy for the first implementation:
- Window size: fixed $5\times5$ LR support
- Expert count: fixed global $K=12$
- Active experts: top-2 per pixel
- Seed source: routed expert seed built by equivariant expert heads
- Selector inputs: invariant summaries only
- Refinement: one shared TP stage after seed formation
- Boundary handling: unlabeled by default, with optional boundary side information as an extension path rather than a core dependency

**Final conclusion**: the complete end-to-end RR-CTP upsampler is now specified as a semiglobal candidate-bank selector with a $5\times5$ hard support, an invariant decision path, an expert-built equivariant seed, and gated equivariant refinement, placed cleanly between the LR feature trunk and the quaternion decoder.
